In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import hashlib
import json
import secrets
import time

# Concept: Proof of Stake (PoS)

Companion to notebook 1 (hash chaining + proof-of-work mining).

**The question PoS answers:** who is allowed to append the next block,
and what stops them from lying — *without* burning electricity on a
nonce race?

| | Proof of Work (notebook 1) | Proof of Stake (this notebook) |
| --- | --- | --- |
| Right to propose | Whoever finds a valid nonce first | Whoever the stake-weighted lottery picks |
| Cost of attacking | Re-mine (CPU / energy) | Risk losing locked **stake** (slashing) |
| Block fields that matter | `nonce`, difficulty target | `proposer`, eligibility at that height |

**What we build here (toy model):**

1. Validators with locked stake
2. Stake-weighted proposer selection
3. Chain validity that checks *historical* eligibility
4. Equivocation detection + slashing (the answer to **nothing-at-stake**)
5. Block rewards that compound stake over time

> **Honest scope:** this is a single-process simulator. Real PoS
> (Ethereum, etc.) also needs slots/epochs, attestations, finality,
> and cryptographic proof that a proposer was really selected (VRF /
> RANDAO). We call out those gaps where they matter — we do not fake them.


## Constants used later (rewards)

Defined up front so Part 2 can credit proposers. Unused until we
re-introduce `Blockchain.add_block()` with earnings.


In [3]:
BLOCK_REWARD: float = 2.0    # newly issued currency paid to the proposer
FEE_PER_TX: float = 0.1      # fee paid by each transaction bundled into the block


## Validators: stake as collateral

In PoW, anyone can mine. In PoS, block proposers are **validators** —
participants who lock currency (**stake**) as skin in the game.

| Idea | Why it matters |
| --- | --- |
| `stake` | Weight in the proposer lottery; also the pile that can be destroyed |
| `slashed_at_height` | When they were caught — so we can judge eligibility *historically* |
| `was_eligible_at(h)` | "Were they clean *at height h*?" — not "are they clean *now*?" |

That last distinction matters: if Alice proposed block #3 honestly, then
cheated at #10, block #3 must stay valid. Punishing her later must not
rewrite earlier history.


In [4]:
class Validator:
    """A network participant who locked stake as collateral to propose blocks.

    Stake is both *influence* (higher stake → higher chance to be picked as
    proposer) and *hostage* (misbehaviour can destroy some or all of it).

    Attributes:
        name: Human-readable identity for demos / logs.
        stake: Locked currency. Grows when rewards are enabled (Part 2).
        slashed_at_height: Height at which this validator was punished, or
            ``None`` if never slashed. Used to judge past eligibility.
    """

    def __init__(self, name: str, stake: float) -> None:
        self.name = name
        self.stake = stake
        self.slashed_at_height: int | None = None

    @property
    def is_slashed(self) -> bool:
        """True if this validator has been slashed at any height."""
        return self.slashed_at_height is not None

    def was_eligible_at(self, height: int) -> bool:
        """Return whether this validator was in good standing at ``height``.

        If never slashed → always eligible.
        If slashed at height S → eligible only for heights strictly before S.
        """
        if self.slashed_at_height is None:
            return True
        return height < self.slashed_at_height

    def __repr__(self) -> str:
        status = (
            f"SLASHED at height {self.slashed_at_height}"
            if self.is_slashed
            else "active"
        )
        return f"{self.name}: stake={self.stake:.2f} [{status}]"


## Anatomy of a PoS block

Same idea as notebook 1 — data + link to parent + hash — but **no mining**:

- No `nonce`
- No difficulty target
- A new field: **`proposer`** — who was allowed to create this block

| Field | Role |
| --- | --- |
| `index` | Position in the chain (`0` = genesis) |
| `timestamp` | When this object was created (informational; also hashed) |
| `data` | Payload (batch of txs in a real chain) |
| `previous_hash` | Hash of the prior block — **the link** |
| `proposer` | Validator name credited with this block |
| `hash` | SHA-256 fingerprint of this block's contents |

Changing any field (including `proposer`) changes `hash`. Changing an
earlier block breaks every later `previous_hash` link — same tamper-
evidence story as PoW, different rule for *who may append*.


In [5]:
class Block:
    """One link in the chain: payload, parent pointer, and who proposed it.

    Attributes:
        index: Position in the chain (0 = genesis).
        timestamp: Creation time (informational; included in the hash).
        data: Payload — a string here; a tx batch in a real chain.
        previous_hash: Hash of the previous block (makes the chain a chain).
        proposer: Name of the validator who proposed this block.
        hash: SHA-256 digest of this block's contents, computed at creation.
    """

    def __init__(self, index: int, data: str, previous_hash: str, proposer: str) -> None:
        self.index = index
        self.timestamp = time.time()
        self.data = data
        self.previous_hash = previous_hash
        self.proposer = proposer
        self.hash = self.compute_hash()

    def compute_hash(self) -> str:
        """Return a deterministic SHA-256 hex digest of this block's fields.

        Same inputs → same output on every machine. That is what lets any
        node re-verify a block without trusting the proposer.
        """
        block_contents = json.dumps(
            {
                "index": self.index,
                "timestamp": self.timestamp,
                "data": self.data,
                "previous_hash": self.previous_hash,
                "proposer": self.proposer,
            },
            sort_keys=True,
        )
        return hashlib.sha256(block_contents.encode()).hexdigest()

    def __repr__(self) -> str:
        return (
            f"Block #{self.index} proposed by {self.proposer}\n"
            f"  data:          {self.data}\n"
            f"  previous_hash: {self.previous_hash[:16]}...\n"
            f"  hash:          {self.hash[:16]}...\n"
        )


## Who proposes next? Stake-weighted lottery

Instead of a hash race, PoS runs a **lottery weighted by stake**:

- Proposer B node with 500 stake is ~5× as likely as Proposer A node with 100 (and ~10× as likely as SketchyGuy's 50)
- Slashed validators are excluded (weight effectively zero)

**Why `secrets` instead of `random`?**  
`random` is a predictable PRNG. If an attacker can forecast the next
proposer, they can DoS that validator right before their slot, or try to
bias selection (**grinding**). Real chains use stronger randomness
(Ethereum: RANDAO + VDF ideas; Algorand: VRF). We use the OS CSPRNG as
a teaching stand-in — still not a full VRF proof attached to the block.

In [6]:
def pick_proposer(validators: list[Validator]) -> Validator:
    """Select the next block proposer at random, weighted by stake.

    Uses ``secrets.randbelow`` (OS CSPRNG) rather than the ``random``
    module, so selection is not trivially predictable from PRNG state.

    Args:
        validators: Full validator set. Slashed validators are skipped.

    Returns:
        The ``Validator`` chosen to propose the next block.

    Notes:
        Integer truncation: ``secrets.randbelow(int(total))`` needs an
        integer total. Fine while stakes are whole numbers. Part 2
        redefines this helper once rewards create fractional stake.
    """
    active = [v for v in validators if not v.is_slashed]
    weights = [v.stake for v in active]
    total = sum(weights)
    pick = secrets.randbelow(int(total))
    cumulative = 0.0
    for validator, weight in zip(active, weights):
        cumulative += weight
        if pick < cumulative:
            return validator
    return active[-1]  # floating-point edge case: land on last active


## Nothing-at-stake, and why we slash

**Nothing-at-stake (the problem):** in naive PoS, producing an extra
conflicting block is nearly free (no mining cost). A rational validator
might sign *every* fork to collect rewards no matter which history wins.

**Slashing (the response):** if you can *prove* misbehaviour — especially
**equivocation** (two different blocks signed for the same height) — the
protocol destroys some or all of that validator's stake.

| Term | Meaning |
| --- | --- |
| Equivocation | Same identity, same height, conflicting block contents |
| Slash | Burn stake + record `slashed_at_height` |
| Intent | Irrelevant to detection — backups that double-sign look identical |

We implement `slash()` here; `detect_equivocation()` lives on
`Blockchain` because it compares *candidates*, not one chain in isolation.


In [7]:
def slash(
    validator: Validator,
    height: int,
    penalty_fraction: float = 1.0,
) -> None:
    """Destroy a fraction of stake and record when the offence occurred.

    Args:
        validator: The validator being punished.
        height: Block height associated with the misbehaviour. Later
            ``was_eligible_at`` / ``is_valid`` use this to keep honest
            *earlier* blocks valid while rejecting *later* ones.
        penalty_fraction: Fraction of stake burned in ``[0, 1]``. Real
            protocols scale this by severity; we often use ``1.0`` for
            demos and a small fraction for "looks accidental" cases.
    """
    lost = validator.stake * penalty_fraction
    validator.stake -= lost
    validator.slashed_at_height = height
    print(
        f"  SLASHED: {validator.name} loses {lost:.0f} coins for "
        f"misbehaviour at height {height}. Remaining stake: {validator.stake:.0f}"
    )


## The chain under PoS rules

Same job as notebook 1's `Blockchain`: own the list of blocks and decide
what is legitimate. Different legitimacy rules:

`is_valid()` checks, for every non-genesis block:

1. **Tamper-evidence** — stored `hash` matches a fresh `compute_hash()`
2. **Linkage** — `previous_hash` equals the previous block's `hash`
3. **Historical eligibility** — the named proposer was a known validator
   who was still eligible *at that block's own height*

`detect_equivocation()` is separate on purpose: one linear chain cannot
see "two blocks for height 6" by itself — you must compare candidates
side by side (as a real network would when gossiping conflicting
proposals).


> **Fix applied to `detect_equivocation`:** the original version only compared
> candidate block hashes, so it could be fed blocks from *different*
> proposers and wrongly call that equivocation. It now raises `ValueError`
> if any candidate does not actually name `proposer_name` — comparing
> different validators' blocks isn't a same-identity double-sign, it's
> just a fork, and the method should refuse to conflate the two. See
> notebook 5 section 5, where this matters: two different validators each
> honestly propose their own block after a network delay, and the fixed
> check is what lets the notebook *prove* that isn't equivocation instead
> of merely asserting it.

In [ ]:
class Blockchain:
    """Owns the chain and enforces PoS-flavoured legitimacy rules.

    Same architectural role as the PoW ``Blockchain`` (append + validate),
    but without a difficulty target. Instead we require a known, eligible
    proposer at each height.
    """

    def __init__(self, validators: list[Validator]) -> None:
        self.validators = validators
        # Genesis has no real proposer; "network" is a sentinel label.
        self.chain: list[Block] = [Block(0, "Genesis Block", "0" * 64, "network")]

    def add_block(self, data: str) -> tuple[Block, Validator]:
        """Run the stake lottery, append the winner's block, return both.

        Args:
            data: Payload for the new block.

        Returns:
            ``(new_block, proposer)``.
        """
        proposer = pick_proposer(self.validators)
        previous_block = self.chain[-1]
        new_block = Block(len(self.chain), data, previous_block.hash, proposer.name)
        self.chain.append(new_block)
        return new_block, proposer

    def propose_candidate(
        self,
        data: str,
        proposer_name: str,
        *,
        previous_hash: str | None = None,
        index: int | None = None,
    ) -> Block:
        """Build a block without appending it, so rival proposals can coexist.

        A fork happens when two proposers each build on the same parent
        before either block has propagated to the other. ``add_block``
        cannot model that -- it always extends ``self.chain`` immediately.
        This method returns a linked ``Block`` and leaves the chain
        untouched, so a caller can build several candidates for the same
        height and compare them (e.g. by attestation weight) before
        ``accept_candidate`` commits one.

        Args:
            data: Payload for the candidate block.
            proposer_name: Validator name proposing this candidate.
            previous_hash: Parent hash to build on. Defaults to the
                current tip's hash; pass an explicit value to model a
                proposer that has not yet seen the latest tip.
            index: Height for the candidate. Defaults to the next height
                after the current tip.

        Returns:
            A new ``Block`` that is not yet part of ``self.chain``.
        """
        tip = self.chain[-1]
        return Block(
            index if index is not None else len(self.chain),
            data,
            previous_hash if previous_hash is not None else tip.hash,
            proposer_name,
        )

    def accept_candidate(self, block: Block) -> None:
        """Append a previously built candidate as the new canonical tip.

        Args:
            block: A candidate produced by ``propose_candidate`` (or an
                equivalent block) whose ``previous_hash`` matches the
                chain's current tip.

        Raises:
            ValueError: If ``block.previous_hash`` does not match the
                current tip's hash -- it would not extend this chain.
        """
        tip = self.chain[-1]
        if block.previous_hash != tip.hash:
            raise ValueError(
                "Candidate does not build on the current chain tip; it "
                "belongs to a different fork or is stale."
            )
        self.chain.append(block)

    def is_valid(self) -> tuple[bool, str]:
        """Re-verify the whole chain from scratch.

        For each block after genesis:
          1. Hash integrity (tamper-evidence)
          2. Parent link matches
          3. Proposer existed and ``was_eligible_at(this_height)``

        Returns:
            ``(True, reason)`` if every check passes, else
            ``(False, first failure reason)``.

        Notes:
            Real PoS also demands cryptographic proof the proposer was
            selected for this slot (VRF / equivalent). This toy only
            checks the *name* against the validator set — flagged, not hidden.
        """
        validators_by_name = {v.name: v for v in self.validators}

        for i in range(1, len(self.chain)):
            current = self.chain[i]
            previous = self.chain[i - 1]

            if current.hash != current.compute_hash():
                return False, f"Block #{current.index} was tampered with directly."

            if current.previous_hash != previous.hash:
                return False, (
                    f"Block #{current.index} is disconnected from "
                    f"Block #{previous.index}."
                )

            proposer = validators_by_name.get(current.proposer)
            if proposer is None:
                return False, (
                    f"Block #{current.index} was proposed by an unknown validator."
                )

            if not proposer.was_eligible_at(current.index):
                return False, (
                    f"Block #{current.index} was proposed by {proposer.name}, who was "
                    f"already slashed (at height {proposer.slashed_at_height}) by the "
                    f"time this block was created."
                )

        return True, "Chain is valid."

    def detect_equivocation(
        self,
        index: int,
        proposer_name: str,
        candidate_blocks: list[Block],
    ) -> tuple[bool, str]:
        """Detect conflicting blocks at the same height from one proposer.

        Equivocation is not a property of a single linear chain; it needs
        two (or more) candidates for the same slot compared side by side.
        Note what equivocation is *not*: two different validators each
        honestly proposing their own block (e.g. an ordinary fork caused
        by network delay) is not equivocation -- only the same identity
        signing multiple conflicting blocks is. This method enforces that
        by requiring every candidate to actually name ``proposer_name``.

        Args:
            index: Height under dispute.
            proposer_name: Validator identity being checked.
            candidate_blocks: All known proposals for that height, which
                must all name ``proposer_name`` as their proposer.

        Returns:
            ``(True, description)`` if more than one distinct hash is present,
            else ``(False, description)``.

        Raises:
            ValueError: If any candidate names a different proposer --
                that is a caller error, not evidence of equivocation:
                conflicting blocks from different proposers are an
                ordinary fork, not one identity double-signing.
        """
        mismatched = {b.proposer for b in candidate_blocks if b.proposer != proposer_name}
        if mismatched:
            raise ValueError(
                "detect_equivocation compares one proposer's own candidates; "
                f"got block(s) proposed by {sorted(mismatched)}, not "
                f"{proposer_name}. Different proposers disagreeing is a "
                "fork, not equivocation."
            )
        hashes = {b.hash for b in candidate_blocks}
        if len(hashes) > 1:
            return True, (
                f"{proposer_name} signed {len(hashes)} conflicting blocks "
                f"at height {index} -- equivocation."
            )
        return False, "No equivocation detected."

`propose_candidate` / `accept_candidate` are not used by the demos below —
they exist so later notebooks can reuse this exact `Blockchain` for a
*fork* (two proposers building on the same tip before either block has
propagated), which `add_block()` alone cannot represent since it always
commits immediately.

---

# Part 1 demos — selection, honesty, then cheating

We set up an unequal stake distribution on purpose so the lottery is
visibly biased toward the largest staker, then show:

1. Honest appends via `add_block()`
2. Equivocation → slash → forged post-slash block rejected
3. Pre-crime history still considered eligible


In [9]:
print("=== Setting up validators ===\n")
validators: list[Validator] = [
    Validator("Proposer-B-Node", stake=500),
    Validator("Proposer-A-Node", stake=100),
    Validator("SketchyGuy-Node", stake=50),
]
for v in validators:
    print(f"  {v}")

chain = Blockchain(validators)


=== Setting up validators ===

  Proposer-B-Node: stake=500.00 [active]
  Proposer-A-Node: stake=100.00 [active]
  SketchyGuy-Node: stake=50.00 [active]


### Demo A — honest rounds

`add_block()` picks a proposer proportional to stake. Over a short run you
will usually see the Proposer B node dominate; smaller stakes still get occasional
slots (randomness, not a round-robin).


In [10]:
print("\n=== Adding 5 blocks honestly, via chain.add_block() ===\n")
for i in range(1, 6):
    block, proposer = chain.add_block(f"Legit transaction batch #{i}")
    print(f"  Block #{block.index} proposed by {proposer.name} (stake={proposer.stake})")

valid, msg = chain.is_valid()
print(f"\nChain valid? {valid} -- {msg}\n")



=== Adding 5 blocks honestly, via chain.add_block() ===

  Block #1 proposed by Proposer-B-Node (stake=500)
  Block #2 proposed by Proposer-B-Node (stake=500)
  Block #3 proposed by Proposer-B-Node (stake=500)
  Block #4 proposed by Proposer-B-Node (stake=500)
  Block #5 proposed by Proposer-B-Node (stake=500)

Chain valid? True -- Chain is valid.



### Demo B — equivocation, slash, then a rejected forgery

SketchyGuy produces **two different blocks for height 6** (classic
double-spend attempt / fork signing). We:

1. Detect equivocation from the candidate pair
2. Slash at height 6 (full stake by default)
3. Append a forged block under their name anyway (bypassing `add_block`
   on purpose — simulating a cheater forcing a write)
4. Watch `is_valid()` fail on historical eligibility
5. Confirm height 5 was still an eligible time for them


In [11]:
print("=== SketchyGuy-Node equivocates at height 6 (two conflicting proposals) ===\n")
sketchy = validators[2]
prev_hash = chain.chain[-1].hash
block_6a = Block(6, "Alice sends Bob $100", prev_hash, sketchy.name)
block_6b = Block(6, "Alice sends CAROL the same $100", prev_hash, sketchy.name)

caught, msg = chain.detect_equivocation(6, sketchy.name, [block_6a, block_6b])
print(f"  {msg}")
if caught:
    slash(sketchy, height=6)


=== SketchyGuy-Node equivocates at height 6 (two conflicting proposals) ===

  SketchyGuy-Node signed 2 conflicting blocks at height 6 -- equivocation.
  SLASHED: SketchyGuy-Node loses 50 coins for misbehaviour at height 6. Remaining stake: 0


In [12]:
print("\n=== SketchyGuy tries to sneak a forged block in anyway, post-slashing ===\n")
forged = Block(
    len(chain.chain),
    "SketchyGuy tries to sneak one in",
    chain.chain[-1].hash,
    sketchy.name,
)
chain.chain.append(forged)  # bypass add_block: attacker writes directly

valid, msg = chain.is_valid()
print(f"  Chain valid? {valid} -- {msg}")



=== SketchyGuy tries to sneak a forged block in anyway, post-slashing ===

  Chain valid? False -- Block #6 was proposed by SketchyGuy-Node, who was already slashed (at height 6) by the time this block was created.


In [13]:
print("\n=== Sanity check: is Block #5 (honest, pre-crime) still OK? ===\n")
block_5 = chain.chain[5]
print(
    f"  Was {sketchy.name} eligible at height {block_5.index}? "
    f"{sketchy.was_eligible_at(block_5.index)}  "
    f"(should be True -- honest history stays honest)"
)

print("\n=== Final validator state ===")
for v in validators:
    print(f"  {v}")



=== Sanity check: is Block #5 (honest, pre-crime) still OK? ===

  Was SketchyGuy-Node eligible at height 5? True  (should be True -- honest history stays honest)

=== Final validator state ===
  Proposer-B-Node: stake=500.00 [active]
  Proposer-A-Node: stake=100.00 [active]
  SketchyGuy-Node: stake=0.00 [SLASHED at height 6]


---

# Part 2 — rewards, compounding, and accidental equivocation

Until now, proposers earned nothing: stake was static. Real PoS pays
proposers so validating is economically rational.

**This section redefines** `pick_proposer`, `Block`, and `Blockchain` in
place (Jupyter keeps the latest definition). Reasons:

| Redefinition | Why |
| --- | --- |
| `pick_proposer` | Fractional millicoin precision once rewards create decimals |
| `Block` | Track `num_transactions` so fees = `num_txs * FEE_PER_TX` |
| `Blockchain.add_block` | Credit `BLOCK_REWARD + fees` straight into proposer stake |

> More stake → picked more often → earn more → even more stake. That
> compounding loop is intentional economics — and a centralisation
> pressure real protocols manage with other mechanisms (we only show the
> raw dynamic).


### Redefinition 1 — lottery that tolerates fractional stake


In [14]:
def pick_proposer(validators: list[Validator]) -> Validator:
    """Stake-weighted lottery with sub-integer precision for fractional stakes.

    Scales stakes by 1000 before ``randbelow``, then compares in float space
    so rewards like ``2.1`` still affect selection odds fairly.
    """
    active = [v for v in validators if not v.is_slashed]
    weights = [v.stake for v in active]
    total = sum(weights)
    pick = secrets.randbelow(int(total * 1000)) / 1000
    cumulative = 0.0
    for validator, weight in zip(active, weights):
        cumulative += weight
        if pick < cumulative:
            return validator
    return active[-1]


### Redefinition 2 — blocks that know how many txs they bundle

`num_transactions` is not magic consensus data; it is how this toy
computes fee income for the proposer.


In [15]:
class Block:
    """One chain link, plus a tx count used only for fee accounting.

    Attributes:
        index: Position in the chain (0 = genesis).
        data: Payload for this block.
        previous_hash: Hash of the previous block.
        proposer: Validator name who proposed this block.
        num_transactions: Count used to compute ``num_transactions * FEE_PER_TX``.
        hash: SHA-256 digest of this block's contents (includes tx count).
    """

    def __init__(
        self,
        index: int,
        data: str,
        previous_hash: str,
        proposer: str,
        num_transactions: int = 1,
    ) -> None:
        self.index = index
        self.timestamp = time.time()
        self.data = data
        self.previous_hash = previous_hash
        self.proposer = proposer
        self.num_transactions = num_transactions
        self.hash = self.compute_hash()

    def compute_hash(self) -> str:
        """Deterministic SHA-256 over this block's fields (including tx count)."""
        block_contents = json.dumps(
            {
                "index": self.index,
                "timestamp": self.timestamp,
                "data": self.data,
                "previous_hash": self.previous_hash,
                "proposer": self.proposer,
                "num_transactions": self.num_transactions,
            },
            sort_keys=True,
        )
        return hashlib.sha256(block_contents.encode()).hexdigest()

    def __repr__(self) -> str:
        return (
            f"Block #{self.index} proposed by {self.proposer} "
            f"({self.num_transactions} txs)\n"
            f"  data: {self.data}\n"
        )


### Redefinition 3 — `add_block` pays the proposer

Earnings land **in stake**, so they immediately change future lottery odds.
That is the entire "rich get richer" mechanic in a few lines.


In [ ]:
class Blockchain:
    """Chain owner with PoS validation rules and proposer rewards."""

    def __init__(self, validators: list[Validator]) -> None:
        self.validators = validators
        self.chain: list[Block] = [Block(0, "Genesis Block", "0" * 64, "network")]

    def add_block(self, data: str, num_transactions: int = 1) -> tuple[Block, Validator]:
        """Select a proposer, append their block, and credit rewards into stake.

        Earnings = ``BLOCK_REWARD + num_transactions * FEE_PER_TX``.

        Args:
            data: Payload for the new block.
            num_transactions: How many txs this block bundles (drives fees).

        Returns:
            ``(new_block, proposer)``.
        """
        proposer = pick_proposer(self.validators)
        previous_block = self.chain[-1]
        new_block = Block(
            len(self.chain), data, previous_block.hash, proposer.name, num_transactions
        )
        self.chain.append(new_block)

        earnings = BLOCK_REWARD + num_transactions * FEE_PER_TX
        proposer.stake += earnings

        return new_block, proposer

    def is_valid(self) -> tuple[bool, str]:
        """Re-verify hash integrity, linkage, and historical proposer eligibility."""
        validators_by_name = {v.name: v for v in self.validators}
        for i in range(1, len(self.chain)):
            current, previous = self.chain[i], self.chain[i - 1]
            if current.hash != current.compute_hash():
                return False, f"Block #{current.index} was tampered with directly."
            if current.previous_hash != previous.hash:
                return False, (
                    f"Block #{current.index} is disconnected from "
                    f"Block #{previous.index}."
                )
            proposer = validators_by_name.get(current.proposer)
            if proposer is None:
                return False, (
                    f"Block #{current.index} was proposed by an unknown validator."
                )
            if not proposer.was_eligible_at(current.index):
                return False, (
                    f"Block #{current.index} was proposed by {proposer.name}, who was "
                    f"already slashed (at height {proposer.slashed_at_height}) by then."
                )
        return True, "Chain is valid."

    def detect_equivocation(
        self,
        index: int,
        proposer_name: str,
        candidate_blocks: list[Block],
    ) -> tuple[bool, str]:
        """Compare candidate hashes at one height; intent is never inspected.

        Protocols cannot read minds — two signatures for conflicting blocks
        are enough evidence, whether the cause was malice or a bad failover.

        Fixed to match Part 1's version: raises if any candidate names a
        proposer other than ``proposer_name`` -- see the note above Part 1's
        `Blockchain` for why that mismatch is a fork, not equivocation.
        """
        mismatched = {b.proposer for b in candidate_blocks if b.proposer != proposer_name}
        if mismatched:
            raise ValueError(
                "detect_equivocation compares one proposer's own candidates; "
                f"got block(s) proposed by {sorted(mismatched)}, not "
                f"{proposer_name}. Different proposers disagreeing is a "
                "fork, not equivocation."
            )
        hashes = {b.hash for b in candidate_blocks}
        if len(hashes) > 1:
            return True, (
                f"{proposer_name} signed {len(hashes)} conflicting blocks "
                f"at height {index} -- equivocation."
            )
        return False, "No equivocation detected."

### Demo C — watch stake compound over 20 rounds

Re-create a fresh validator set (Part 1 may have slashed SketchyGuy).
Print snapshots every 5 rounds and final share of total stake.


In [17]:
print("=== Demo C: watching stake compound over 20 honest rounds ===\n")
validators = [
    Validator("Proposer-B-Node", stake=500),
    Validator("Proposer-A-Node", stake=100),
    Validator("SketchyGuy-Node", stake=50),
]
chain = Blockchain(validators)


=== Demo C: watching stake compound over 20 honest rounds ===



In [18]:
for i in range(1, 21):
    num_txs = secrets.randbelow(20) + 1
    chain.add_block(f"Batch #{i}", num_transactions=num_txs)
    if i % 5 == 0:
        print(f"  -- after {i} rounds --")
        for v in validators:
            print(f"     {v}")
        print()

total_stake = sum(v.stake for v in validators)
print("Final stake share after 20 rounds:")
for v in validators:
    print(f"  {v.name}: {v.stake:.2f}  ({100 * v.stake / total_stake:.1f}% of total)")
print("(Watch who pulled further ahead: more stake -> picked more often -> earns")
print(" more -> even more stake next time. Compounding, exactly like a bank account.)\n")


  -- after 5 rounds --
     Proposer-B-Node: stake=511.90 [active]
     Proposer-A-Node: stake=103.30 [active]
     SketchyGuy-Node: stake=50.00 [active]

  -- after 10 rounds --
     Proposer-B-Node: stake=525.00 [active]
     Proposer-A-Node: stake=107.30 [active]
     SketchyGuy-Node: stake=50.00 [active]

  -- after 15 rounds --
     Proposer-B-Node: stake=541.60 [active]
     Proposer-A-Node: stake=107.30 [active]
     SketchyGuy-Node: stake=50.00 [active]

  -- after 20 rounds --
     Proposer-B-Node: stake=556.30 [active]
     Proposer-A-Node: stake=107.30 [active]
     SketchyGuy-Node: stake=50.00 [active]

Final stake share after 20 rounds:
  Proposer-B-Node: 556.30  (78.0% of total)
  Proposer-A-Node: 107.30  (15.0% of total)
  SketchyGuy-Node: 50.00  (7.0% of total)
(Watch who pulled further ahead: more stake -> picked more often -> earns
 more -> even more stake next time. Compounding, exactly like a bank account.)



### Demo D — honest operator, still slashable

**Scenario:** one validator key on primary + "backup" machine (common HA
instinct). Primary drops; backup proposes; primary reconnects and also
proposes for the **same height**. Neither meant to cheat.

`detect_equivocation()` does not ask *why* — only whether two conflicting
hashes exist. We apply a **small** `penalty_fraction` to echo how real
protocols often treat a first / apparently accidental offence more gently
than coordinated attacks.


In [19]:
print("=== Demo D: an HONEST validator accidentally equivocates ===\n")
validators2 = [Validator("Titus-Home-Validator", stake=200)]
chain2 = Blockchain(validators2)
for i in range(1, 4):
    chain2.add_block(f"Legit batch #{i}")

print("  Scenario: runs a 'backup' node for reliability -- same validator key")
print("  on both machines, standard HA instinct. The primary briefly drops off")
print("  the network; the backup takes over and proposes a block. A moment")
print("  later the primary reconnects, unaware a failover happened, and ALSO")
print("  proposes for the same height. Neither machine did anything malicious.\n")

solo = validators2[0]
prev_hash = chain2.chain[-1].hash
height = len(chain2.chain)
block_primary = Block(height, "Primary node's version of events", prev_hash, solo.name)
block_backup = Block(height, "Backup node's version of events", prev_hash, solo.name)

caught, msg = chain2.detect_equivocation(height, solo.name, [block_primary, block_backup])
print(f"  {msg}")
print("  detect_equivocation() never asked WHY there were two signatures.")
print("  It only sees: same identity, same height, conflicting content. Guilty.\n")

if caught:
    # Small fraction: echo gentler initial penalties used in real protocols
    # for first / apparently-accidental offences (not a full wipe).
    slash(solo, height=height, penalty_fraction=0.02)


=== Demo D: an HONEST validator accidentally equivocates ===

  Scenario: runs a 'backup' node for reliability -- same validator key
  on both machines, standard HA instinct. The primary briefly drops off
  the network; the backup takes over and proposes a block. A moment
  later the primary reconnects, unaware a failover happened, and ALSO
  proposes for the same height. Neither machine did anything malicious.

  Titus-Home-Validator signed 2 conflicting blocks at height 4 -- equivocation.
  detect_equivocation() never asked WHY there were two signatures.
  It only sees: same identity, same height, conflicting content. Guilty.

  SLASHED: Titus-Home-Validator loses 4 coins for misbehaviour at height 4. Remaining stake: 202


In [20]:
print(f"\n  Final state: {solo}")



  Final state: Titus-Home-Validator: stake=202.17 [SLASHED at height 4]


---

## Takeaways

1. **PoS replaces energy cost with economic collateral** — influence and
   liability both scale with stake.
2. **Proposer selection is a weighted lottery**, not a hash puzzle; randomness
   quality matters (predictability enables grinding / targeted DoS).
3. **Nothing-at-stake** is why conflicting signatures must be expensive —
   slashing turns equivocation into a real loss.
4. **Validate eligibility historically** — slash later without invalidating
   honest earlier blocks.
5. **Rewards compound stake** — economically expected, and a centralisation
   pressure to remember.
6. **Detection ignores intent** — a bad failover looks like an attack.

**Still out of scope (on purpose):** peer networking, committees/attestations,
finality gadgets, withdrawal delays, and cryptographic selection proofs
(VRF / RANDAO). Add those when you leave the toy and read a real protocol
spec.
